# Week 5b -- Design Trends Analysis

This notebook analyzes `design_trends.csv`, which was generated by `design_trends.py` using the Wikipedia Pageviews API. Each row represents one HCI or UX-related Wikipedia article and summarizes 90 days of daily traffic data from January 30 to April 30, 2026.

The goal is to understand which topics are genuinely rising in public interest, which have peaked, and which are fading -- using traffic as a proxy for cultural attention.

**Before running:** make sure `design_trends.csv` is in the same folder as this notebook. If not, run `python design_trends.py` first.

## Setup

In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(".")
TRENDS_CSV = DATA_DIR / "design_trends.csv"

assert TRENDS_CSV.is_file(), (
    f"Missing {TRENDS_CSV.resolve()}. Run: python design_trends.py"
)

print("pandas:", pd.__version__)

pandas: 3.0.2


---
## 1. What does the dataset look like?

`head()` shows the first few rows so we can see what fields exist and what the top-ranked articles are. `info()` tells us the data types and whether any columns have missing values.

In [3]:
df = pd.read_csv(TRENDS_CSV, encoding="utf-8")

# The top rows are the most-viewed HCI topics on Wikipedia in the last 90 days.
# Rank 1 having the most views tells us what the public is most curious about
# in the HCI and UX space right now.
df.head(10)

,rank,trend,window_start,window_end,days_with_data,total_views_in_window,avg_views_per_day,est_views_per_month,peak_date,peak_views,peak_to_mean_daily_ratio,recent_30d_avg_views,prior_30d_avg_views,pct_change_recent_vs_prior_30d,momentum,trend_status,lcd_line1,lcd_line2
0,1,BonziBuddy,2026-01-30,2026-04-30,90,46283,514.26,15652.3,2026-02-26,948,1.84,486.00,580.67,-16.3,Fading,fading,BonziBuddy,Status: Fading
1,2,Brain–computer interface,2026-01-30,2026-04-30,90,39797,442.19,13458.8,2026-02-09,598,1.35,399.03,458.57,-13.0,Fading,fading,Brain–computer i,Status: Fading
2,3,Human–computer interaction,2026-01-30,2026-04-30,90,21997,244.41,7439.1,2026-02-27,336,1.37,231.93,238.03,-2.6,Peaked / Stable,peaked_or_stable,Human–computer i,Status: Peaked /
3,4,Center for Humane Technology,2026-01-30,2026-04-30,90,8973,99.70,3034.6,2026-03-22,457,4.58,115.03,117.83,-2.4,Peaked / Stable,peaked_or_stable,Center for Human,Status: Peaked /
4,5,As We May Think,2026-01-30,2026-04-30,90,7783,86.48,2632.1,2026-02-28,275,3.18,80.70,84.30,-4.3,Peaked / Stable,peaked_or_stable,As We May Think,Status: Peaked /
5,6,Bad Day (viral video),2026-01-30,2026-04-30,90,6310,70.11,2134.0,2026-04-29,1109,15.82,103.37,51.63,100.2,Rising,rising,Bad Day (viral v,Status: Rising
6,7,10-foot user interface,2026-01-30,2026-04-30,90,3888,43.20,1314.9,2026-02-02,68,1.57,40.73,42.20,-3.5,Peaked / Stable,peaked_or_stable,10-foot user int,Status: Peaked /
7,8,Computers are social actors,2026-01-30,2026-04-30,90,3345,37.17,1131.2,2026-04-27,70,1.88,38.40,38.00,1.1,Peaked / Stable,peaked_or_stable,Computers are so,Status: Peaked /
8,9,Conference on Human Factors in Computing Systems,2026-01-30,2026-04-30,90,3221,35.79,1089.3,2026-04-13,76,2.12,39.77,33.70,18.0,Rising,rising,Conference on Hu,Status: Rising
9,10,Addiction by Design,2026-01-30,2026-04-30,90,2763,30.70,934.4,2026-02-10,104,3.39,30.90,25.97,19.0,Rising,rising,Addiction by Des,Status: Rising


In [4]:
# info() confirms column types and whether anything is missing.
# If a column shows fewer non-null values than the total rows,
# that means some articles did not have enough data for that calculation.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   rank                            50 non-null     int64  
 1   trend                           50 non-null     str    
 2   window_start                    50 non-null     str    
 3   window_end                      50 non-null     str    
 4   days_with_data                  50 non-null     int64  
 5   total_views_in_window           50 non-null     int64  
 6   avg_views_per_day               50 non-null     float64
 7   est_views_per_month             50 non-null     float64
 8   peak_date                       50 non-null     str    
 9   peak_views                      50 non-null     int64  
 10  peak_to_mean_daily_ratio        50 non-null     float64
 11  recent_30d_avg_views            50 non-null     float64
 12  prior_30d_avg_views             50 non-null     f

---
## 2. How is traffic distributed across all 50 articles?

`describe()` gives us the summary statistics for total views. This tells us whether traffic is spread evenly or concentrated in a few top articles.

In [5]:
# The gap between the mean and the max tells us how concentrated traffic is.
# If the max is much higher than the mean, a small number of articles
# are pulling most of the public attention -- which is typical of how
# cultural interest works. Most topics get ignored; a few dominate.
df["total_views_in_window"].describe()

count       50.000000
mean      3649.200000
std       8845.247637
min         77.000000
25%        354.500000
50%       1123.000000
75%       2644.250000
max      46283.000000
Name: total_views_in_window, dtype: float64

In [6]:
# value_counts() on the momentum column shows how many articles
# are rising, peaked/stable, or fading right now.
# This gives a quick read on the overall health of public interest
# in HCI topics -- is curiosity growing or declining as a whole?
df["momentum"].value_counts()

momentum
Peaked / Stable    23
Fading             19
Rising              8
Name: count, dtype: int64

---
## 3. Which topics are currently rising in public interest?

We filter to only the articles with Rising momentum -- meaning their traffic in the most recent 30 days was at least 10% higher than the 30 days before that. These are the topics the public is becoming more curious about right now.

In [7]:
# Filtering to Rising articles only.
# The pct_change column tells us exactly how much traffic grew.
# A topic with 100% change means it doubled in views -- that is a strong signal,
# not just noise. A topic with 10-15% change is growing but modestly.
rising = df[df["momentum"] == "Rising"].copy()

print(f"{len(rising)} out of {len(df)} tracked articles are currently Rising.")
rising[["rank", "trend", "total_views_in_window", "pct_change_recent_vs_prior_30d"]].sort_values(
    "pct_change_recent_vs_prior_30d", ascending=False
)

8 out of 50 tracked articles are currently Rising.


,rank,trend,total_views_in_window,pct_change_recent_vs_prior_30d
5,6,Bad Day (viral video),6310,100.2
47,48,Alerting system,88,60.0
43,44,Confederate effect,169,23.9
13,14,3Dconnexion,2633,19.9
9,10,Addiction by Design,2763,19.0
8,9,Conference on Human Factors in Computing Systems,3221,18.0
21,22,Cognitive engineering,1175,15.1
35,36,Adaptation (computer science),406,11.8


---
## 4. Does momentum correlate with total traffic volume?

We group by momentum category and take the mean of total views. This answers: are rising topics generally the most-viewed ones, or can a low-traffic topic still be growing?

In [8]:
# Grouping by momentum and averaging total views.
# If Rising articles have lower average views than Fading ones,
# it means new topics are emerging from obscurity -- which is interesting
# because it suggests the next big thing in HCI discourse might currently
# have very low absolute traffic but be growing fast.
mean_views_by_momentum = (
    df.groupby("momentum", as_index=False)["total_views_in_window"]
    .mean()
    .rename(columns={"total_views_in_window": "mean_total_views"})
    .sort_values("mean_total_views", ascending=False)
)
mean_views_by_momentum

,momentum,mean_total_views
0,Fading,5233.894737
1,Peaked / Stable,2880.478261
2,Rising,2095.625000


---
## 5. Are there any missing values or data quality issues?

Some articles may not have had enough days of data for the momentum calculation to run. We check which columns are incomplete and what that means for the analysis.

In [9]:
# isnull().sum() counts how many rows are missing a value in each column.
# If pct_change or recent_30d_avg_views has missing values, it means
# those articles did not have enough data for a 60-day comparison.
# That is a real data quality issue -- it means the momentum label
# for those rows may not be reliable.
missing = df.isnull().sum()
print("Columns with missing values:")
print(missing[missing > 0])
print()
print("Total rows:", len(df))
print("All columns complete:", missing.sum() == 0)

Columns with missing values:
Series([], dtype: int64)

Total rows: 50
All columns complete: True


In [10]:
# Check which specific articles have fewer than 90 days of data.
# An article with only 80 days means Wikipedia had no recorded views
# on some days -- possibly because the page had zero traffic,
# not because the data is broken.
incomplete = df[df["days_with_data"] < 90][["rank", "trend", "days_with_data", "momentum"]]
print(f"{len(incomplete)} articles have fewer than 90 days of data:")
incomplete

10 articles have fewer than 90 days of data:


,rank,trend,days_with_data,momentum
29,30,BioWall,80,Fading
39,40,Cortical modem,89,Peaked / Stable
42,43,Brain painting,88,Peaked / Stable
43,44,Confederate effect,88,Rising
44,45,Ben Bederson,87,Peaked / Stable
45,46,Computer Graphics International,86,Peaked / Stable
46,47,CMN-GOMS,85,Fading
47,48,Alerting system,78,Rising
48,49,Barrier pointing,83,Fading
49,50,Cognitive infocommunications,83,Fading
